# Aetheris Pre-Operative Risk ML Model Training

This notebook trains a Random Forest Classifier on the `MImic IV Dataset.xlsx` to predict 30-day post-operative complications (`Complication 30d (0/1)`). We will encode categorical variables, scale numerical features, and export the trained model along with the preprocessor to the `aetheris-backend/app/ml/` directory.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

## 1. Load the Dataset
We load the Excel dataset, skipping the title row.

In [ ]:
df = pd.read_excel('../MImic IV Dataset.xlsx', header=1)
print("Dataset Shape:", df.shape)
df.head()

## 2. Preprocessing
Select relevant clinical features. We drop high-cardinality or target-leakage columns like IDs or computed Risk Scores/Categories.

In [ ]:
# Define Target
TARGET = 'Complication 30d (0/1)'

# Define Features
NUMERICAL = ['Age (yrs)', 'Weight (kg)', 'Height (cm)', 'BMI', 
             'Systolic BP', 'Diastolic BP', 'Creatinine (mg/dL)', 
             'eGFR (mL/min)', 'Hemoglobin (g/dL)', 'Prior Surgery Count', 'Med Count']

CATEGORICAL = ['Gender', 'Diabetes (0/1)', 'Hypertension (0/1)', 'ASA Class']

X = df[NUMERICAL + CATEGORICAL].copy()
y = df[TARGET]

# Convert binary ints masquerading as strings if necessary
X['Diabetes (0/1)'] = X['Diabetes (0/1)'].astype(str)
X['Hypertension (0/1)'] = X['Hypertension (0/1)'].astype(str)

# Handle Missing Values inline
X[NUMERICAL] = X[NUMERICAL].fillna(X[NUMERICAL].median())
X[CATEGORICAL] = X[CATEGORICAL].fillna('Unknown')

## 3. Build Training Pipeline
We use a `ColumnTransformer` to scale numbers and one-hot encode categories.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)
    ])

# We use Random Forest to get probability scores for risk assessment
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))])

## 4. Train and Evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

## 5. Export Model for Backend
Save the pipeline to `aetheris-backend/app/ml/mimic_preop_model.pkl` so FastAPI can load it.

In [ ]:
os.makedirs('../aetheris-backend/app/ml', exist_ok=True)
model_path = '../aetheris-backend/app/ml/mimic_preop_model.pkl'
joblib.dump(pipeline, model_path)
print(f"Model successfully exported to {model_path}")